# A surface that follows the formula

`plotly.graph_objects.FigureWidget` is a widget in its own right: assigning to
`fig.data[0].z` moves the surface in the browser, and `uirevision` leaves the
camera where the reader put it.  Beside the editor, that makes a formula in
two variables something you can turn around while you edit it.

Needs `plotly` (`pip install plotly`) and `numpy` beside
`sympy-editor[jupyter]`.

In [ ]:
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from sympy import lambdify, symbols

from sympy_editor import edit

x, y = symbols("x y")

## The formula on a grid

The same `lambdify` as in `plot_alongside.ipynb`, over a mesh instead of a
line: whatever is not a real number there becomes a `NaN`, and plotly leaves
a hole in the surface where those are.  A pole is not a hole, though — it is
a spike that would flatten everything else — so the vertical range is taken
from the bulk of the samples, as the curve's was.

In [ ]:
def grid(expr, span=3.0, n=60):
    """`expr` sampled on an n x n square of side 2*span, centred on the origin."""
    axis = np.linspace(-span, span, n)
    xs, ys = np.meshgrid(axis, axis)
    f = lambdify((x, y), expr, "numpy")
    with np.errstate(all="ignore"):
        zs = np.asarray(f(xs, ys), dtype=complex) + np.zeros_like(xs)   # a constant broadcasts
    zs = np.where(np.abs(zs.imag) < 1e-9, zs.real, np.nan)
    return axis, np.where(np.isfinite(zs), zs, np.nan)                  # 1/0 is a hole


def range_of(zs):
    """A vertical range the bulk of the surface fits in, poles or no poles."""
    finite = zs[np.isfinite(zs)]
    edge = float(np.percentile(np.abs(finite), 98)) * 1.2 if finite.size else 1.0
    return (-edge, edge) if edge else (-1.0, 1.0)

## The two widgets

`batch_update` sends one message to the browser instead of three, so the
surface and its title change together.  A formula that cannot be sampled —
half-typed, or in the wrong variables — says so in the title and leaves the
last surface standing.

In [ ]:
span, mesh = 3.0, 60
start = x**2 - y**2

axis, zs = grid(start, span, mesh)
fig = go.FigureWidget(go.Surface(x=axis, y=axis, z=zs, colorscale="Viridis", showscale=False))
fig.update_layout(height=430, margin=dict(l=0, r=0, t=34, b=0), title=str(start),
                  uirevision="keep",           # a redraw leaves the camera alone
                  scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="",
                             zaxis=dict(range=range_of(zs))))

w = edit(start)


@w.on_change
def redraw(expr):
    try:
        axis, zs = grid(expr, span, mesh)
    except Exception as exc:
        fig.layout.title.text = f"{expr}  —  {type(exc).__name__}: {exc}"
        return
    with fig.batch_update():
        fig.data[0].x, fig.data[0].y, fig.data[0].z = axis, axis, zs
        fig.layout.scene.zaxis.range = range_of(zs)
        fig.layout.title.text = str(expr)


widgets.VBox([w, fig])

## How much of it to look at

Two more widgets on the same picture, and none of them on the formula: the
span is how far out the grid reaches, the mesh how finely it is sampled.

In [ ]:
span_slider = widgets.FloatSlider(value=span, min=1.0, max=8.0, step=0.5, description="span")
mesh_slider = widgets.IntSlider(value=mesh, min=20, max=140, step=10, description="mesh")


def resample(*_):
    global span, mesh
    span, mesh = span_slider.value, mesh_slider.value
    redraw(w.expr)


span_slider.observe(resample, "value")
mesh_slider.observe(resample, "value")
widgets.HBox([span_slider, mesh_slider])

### What to try

* Select the whole formula and **Transform ▾ → diff** (in `x`): the saddle
  flattens into a plane, and the camera does not move.
* `exp(-(x**2 + y**2))`, `sin(x)*cos(y)`, `x*y/(x**2 + y**2 + 1)`.
* Turn the surface, then edit: what you are looking at is yours to keep, and
  only the shape under it changes.
* `1/(x**2 + y**2)` climbs to a pole at the origin: the spike is cut off at
  the top rather than pressing the rest of the surface flat.